<a href="https://colab.research.google.com/github/Izzatbek2011/My/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import asyncio
import re
import io
import uuid
import json
import os
import nest_asyncio

from aiogram import Bot, Dispatcher, F
from aiogram.filters import Command
from aiogram.fsm.context import FSMContext
from aiogram.fsm.state import State, StatesGroup
from aiogram.types import (
    Message,
    CallbackQuery,
    InlineKeyboardButton,
    InlineKeyboardMarkup,
    ReplyKeyboardMarkup,
    KeyboardButton,
)

import PyPDF2
import docx
from PIL import Image
import pytesseract

nest_asyncio.apply()

BOT_TOKEN = "8825713233:AAHT6kT60fnmJVuXbPAO49OCrBzYxrkZ9JM"

bot = Bot(token=BOT_TOKEN)
dp = Dispatcher()

DATA_FILE = "global_tests.json"

# ==========================================
# MA'LUMOTLARNI FAYLGA SAQLASH VA O'QISH
# ==========================================

def load_tests():
    if os.path.exists(DATA_FILE):
        try:
            with open(DATA_FILE, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return {}
    return {}

def save_tests(data):
    with open(DATA_FILE, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

global_tests = load_tests()
active_quizzes = {}


class TestState(StatesGroup):
    entering_set_name = State()
    entering_bulk_questions = State()
    choosing_correct = State()
    answering = State()


def main_menu():
    return ReplyKeyboardMarkup(
        keyboard=[
            [KeyboardButton(text="➕ Yangi to'plam yaratish"), KeyboardButton(text="📝 Testni boshlash")],
            [KeyboardButton(text="📊 Mening testlarim"), KeyboardButton(text="🗑 Testlarimni o'chirish")]
        ],
        resize_keyboard=True
    )


def correct_choice_keyboard():
    return InlineKeyboardMarkup(
        inline_keyboard=[
            [
                InlineKeyboardButton(text="A", callback_data="correct_A"),
                InlineKeyboardButton(text="B", callback_data="correct_B"),
                InlineKeyboardButton(text="C", callback_data="correct_C"),
                InlineKeyboardButton(text="D", callback_data="correct_D"),
            ]
        ]
    )


def parse_questions_from_text(text: str):
    parsed_questions = []
    cleaned_text = re.sub(r'\s+', ' ', text)
    blocks = re.split(r'(?=\b\d+[\.\)]\s*)', cleaned_text)

    for block in blocks:
        block = block.strip()
        if not block:
            continue

        pattern = (
            r'(?:^\d+[\.\)]\s*)?(?P<q>.+?)'
            r'\s+[Aa][\)\.\:\-]?\s*(?P<a>.+?)'
            r'\s+[Bb][\)\.\:\-]?\s*(?P<b>.+?)'
            r'\s+[Cc][\)\.\:\-]?\s*(?P<c>.+?)'
            r'\s+[Dd][\)\.\:\-]?\s*(?P<d>.+)'
        )
        match = re.search(pattern, block)

        if match:
            q_data = match.groupdict()
            question_text = q_data["q"].strip()
            ans_a = q_data["a"].strip()
            ans_b = q_data["b"].strip()
            ans_c = q_data["c"].strip()
            ans_d_full = q_data["d"].strip()

            correct_match = re.search(r'(?:Javob|To\'g\'ri javob|Kalit)[\s\:\-]*([A-Da-d])', ans_d_full, re.IGNORECASE)

            correct_ans = None
            if correct_match:
                correct_ans = correct_match.group(1).upper()
                ans_d = re.sub(r'(?:Javob|To\'g\'ri javob|Kalit)[\s\:\-]*[A-Da-d].*', '', ans_d_full, flags=re.IGNORECASE).strip()
            else:
                ans_d = ans_d_full

            parsed_questions.append({
                "question": question_text,
                "A": ans_a,
                "B": ans_b,
                "C": ans_c,
                "D": ans_d,
                "correct": correct_ans
            })

    return parsed_questions


async def extract_text_from_document(message: Message) -> str:
    document = message.document
    file_name = document.file_name.lower()

    file_bytes = io.BytesIO()
    await bot.download(document, destination=file_bytes)
    file_bytes.seek(0)

    text = ""
    if file_name.endswith('.txt'):
        text = file_bytes.read().decode('utf-8', errors='ignore')
    elif file_name.endswith('.pdf'):
        pdf_reader = PyPDF2.PdfReader(file_bytes)
        for page in pdf_reader.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted + "\n"
    elif file_name.endswith('.docx'):
        doc = docx.Document(file_bytes)
        text = "\n".join([p.text for p in doc.paragraphs])

    return text


async def extract_text_from_photo(message: Message) -> str:
    photo = message.photo[-1]
    file_bytes = io.BytesIO()
    await bot.download(photo, destination=file_bytes)
    file_bytes.seek(0)

    image = Image.open(file_bytes)
    text = pytesseract.image_to_string(image, lang='uzb+eng+rus')
    return text


async def send_question_with_timer(user_id, state: FSMContext):
    quiz = active_quizzes.get(user_id)
    if not quiz:
        return

    current_idx = quiz["current"]
    questions = quiz["questions"]

    if current_idx >= len(questions):
        await show_final_results(user_id, state)
        return

    q = questions[current_idx]

    keyboard = InlineKeyboardMarkup(
        inline_keyboard=[
            [InlineKeyboardButton(text=f"A) {q['A']}", callback_data="ans_A")],
            [InlineKeyboardButton(text=f"B) {q['B']}", callback_data="ans_B")],
            [InlineKeyboardButton(text=f"C) {q['C']}", callback_data="ans_C")],
            [InlineKeyboardButton(text=f"D) {q['D']}", callback_data="ans_D")],
            [InlineKeyboardButton(text="🛑 Testni yakunlash", callback_data="stop_quiz")]
        ]
    )

    msg = await bot.send_message(
        user_id,
        f"📌 <b>{current_idx + 1}-savol / {len(questions)}</b>\n\n"
        f"❓ {q['question']}\n\n"
        f"⏱ <b>Vaqt: 60 sekund</b>",
        reply_markup=keyboard,
        parse_mode="HTML"
    )

    quiz["timer_task"] = asyncio.create_task(question_timer(user_id, current_idx, msg.message_id, state))


async def question_timer(user_id, question_idx, message_id, state: FSMContext):
    await asyncio.sleep(60)

    quiz = active_quizzes.get(user_id)
    if quiz and quiz["current"] == question_idx:
        quiz["wrong"] += 1
        quiz["current"] += 1

        try:
            await bot.edit_message_text(
                chat_id=user_id,
                message_id=message_id,
                text="⏰ <b>Vaqt tugadi!</b> Javob berilmadi.",
                parse_mode="HTML"
            )
        except Exception:
            pass

        await asyncio.sleep(1)
        await send_question_with_timer(user_id, state)


async def show_final_results(user_id, state: FSMContext, is_stopped=False):
    quiz = active_quizzes.get(user_id)
    if not quiz:
        return

    if quiz.get("timer_task"):
        quiz["timer_task"].cancel()

    total = len(quiz["questions"])
    correct = quiz["correct"]
    wrong = quiz["wrong"]
    answered = correct + wrong
    percentage = round((correct / total) * 100) if total > 0 else 0

    status_title = "🛑 <b>TEST MUDDATIDAN OLDIN TO'XTATILDI!</b>" if is_stopped else "🏁 <b>TEST YAKUNLANDI!</b>"

    await bot.send_message(
        user_id,
        f"{status_title}\n\n"
        f"📊 Jami savollar: {total} ta\n"
        f"📝 Javob berilgan: {answered} ta\n"
        f"✅ To'g'ri javoblar: {correct} ta\n"
        f"❌ Noto'g'ri/Vaqt o'tdi: {wrong} ta\n"
        f"📈 Umumiy natija: <b>{percentage}%</b>",
        reply_markup=main_menu(),
        parse_mode="HTML"
    )

    active_quizzes.pop(user_id, None)
    await state.clear()


@dp.message(Command("start"))
async def start_cmd(message: Message, state: FSMContext):
    await state.clear()
    await message.answer(
        "👋 <b>Xush kelibsiz!</b>\n\n"
        "Barcha foydalanuvchilarning testlarini yechishingiz yoki yangi test qo'shishingiz mumkin.",
        reply_markup=main_menu(),
        parse_mode="HTML"
    )


# ==========================================
# TEST TO'PLAMINI YARATISH QISMI
# ==========================================

@dp.message(F.text == "➕ Yangi to'plam yaratish")
async def add_set_start(message: Message, state: FSMContext):
    await state.set_state(TestState.entering_set_name)
    await message.answer(
        "🏷 <b>Yangi test to'plami nomini kiriting:</b>\n"
        "<i>(Masalan: Tarix 1-bob, Fizika 9-sinf)</i>",
        parse_mode="HTML"
    )


@dp.message(TestState.entering_set_name)
async def process_set_name(message: Message, state: FSMContext):
    set_name = message.text.strip()
    test_code = str(uuid.uuid4())[:8]

    await state.update_data(set_name=set_name, test_code=test_code, questions=[])
    await state.set_state(TestState.entering_bulk_questions)

    await message.answer(
        f"📁 To'plam nomi: <b>{set_name}</b>\n\n"
        "Endi testlarni yuboring (Matn, TXT, PDF, DOCX yoki Rasm ko'rinishida):",
        parse_mode="HTML"
    )


@dp.message(TestState.entering_bulk_questions, F.content_type.in_({'text', 'document', 'photo'}))
async def process_bulk_input(message: Message, state: FSMContext):
    user_id = message.from_user.id
    raw_text = ""

    if message.document:
        status_msg = await message.answer("📄 Fayl o'qilmoqda...")
        raw_text = await extract_text_from_document(message)
        await status_msg.delete()
    elif message.photo:
        status_msg = await message.answer("🖼 Rasmdagi matn ajratilmoqda...")
        raw_text = await extract_text_from_photo(message)
        await status_msg.delete()
    else:
        raw_text = message.text.strip()

    if not raw_text:
        await message.answer("❌ Matn ajratib bo'lmadi.")
        return

    parsed = parse_questions_from_text(raw_text)

    if not parsed:
        await message.answer("❌ Savollar topilmadi. Variantlar (A, B, C, D) borligini tekshiring.")
        return

    data = await state.get_data()
    set_name = data.get("set_name", "Umumiy testlar")
    test_code = data.get("test_code")
    questions = data.get("questions", [])

    ready_questions = [q for q in parsed if q["correct"] is not None]
    pending_questions = [q for q in parsed if q["correct"] is None]

    questions.extend(ready_questions)
    await state.update_data(questions=questions)

    if pending_questions:
        await state.update_data(pending=pending_questions, pending_idx=0)
        await state.set_state(TestState.choosing_correct)

        q = pending_questions[0]
        await message.answer(
            f"❓ <b>Savol:</b> {q['question']}\n\n"
            f"A) {q['A']}\nB) {q['B']}\nC) {q['C']}\nD) {q['D']}\n\n"
            "✅ <b>To'g'ri javobni tanlang:</b>",
            reply_markup=correct_choice_keyboard(),
            parse_mode="HTML"
        )
    else:
        global_tests[test_code] = {
            "name": set_name,
            "author": user_id,
            "questions": questions
        }
        save_tests(global_tests)  # Faylga saqlash
        await state.clear()
        await message.answer(
            f"✅ <b>'{set_name}' to'plami umumiy bazaga saqlandi!</b>\n"
            f"Endi ushbu testni barcha foydalanuvchilar yechishi mumkin.",
            reply_markup=main_menu(),
            parse_mode="HTML"
        )


@dp.callback_query(TestState.choosing_correct, F.data.startswith("correct_"))
async def process_correct_choice(callback: CallbackQuery, state: FSMContext):
    correct_ans = callback.data.split("_")[1]
    data = await state.get_data()
    user_id = callback.from_user.id
    set_name = data.get("set_name")
    test_code = data.get("test_code")
    questions = data.get("questions", [])

    pending = data["pending"]
    idx = data["pending_idx"]

    q = pending[idx]
    q["correct"] = correct_ans
    questions.append(q)

    idx += 1

    if idx < len(pending):
        await state.update_data(pending_idx=idx, questions=questions)
        next_q = pending[idx]
        await callback.message.edit_text(
            f"❓ <b>Savol:</b> {next_q['question']}\n\n"
            f"A) {next_q['A']}\nB) {next_q['B']}\nC) {next_q['C']}\nD) {next_q['D']}\n\n"
            "✅ <b>To'g'ri javobni tanlang:</b>",
            reply_markup=correct_choice_keyboard(),
            parse_mode="HTML"
        )
    else:
        global_tests[test_code] = {
            "name": set_name,
            "author": user_id,
            "questions": questions
        }
        save_tests(global_tests)  # Faylga saqlash
        await state.clear()
        await callback.message.edit_text(
            f"✅ <b>Barcha savollar umumiy bazaga saqlandi!</b>",
            parse_mode="HTML"
        )
        await callback.message.answer("Testlarni boshlashingiz mumkin.", reply_markup=main_menu())

    await callback.answer()


@dp.message(F.text == "📊 Mening testlarim")
async def show_my_tests(message: Message):
    user_id = message.from_user.id
    my_tests = {code: data for code, data in global_tests.items() if data["author"] == user_id}

    if not my_tests:
        await message.answer("📋 Siz hali test yaratmagansiz.", reply_markup=main_menu())
        return

    text = "📋 <b>Siz yaratgan testlar:</b>\n\n"
    for code, data in my_tests.items():
        text += f"🔹 <b>{data['name']}</b> — {len(data['questions'])} ta savol\n"

    await message.answer(text, reply_markup=main_menu(), parse_mode="HTML")


@dp.message(F.text == "🗑 Testlarimni o'chirish")
async def clear_tests(message: Message):
    user_id = message.from_user.id
    to_delete = [code for code, data in global_tests.items() if data["author"] == user_id]

    for code in to_delete:
        del global_tests[code]

    save_tests(global_tests)  # O'chirilgandan so'ng qayta saqlash

    await message.answer("🗑 Siz yaratgan testlar o'chirildi.", reply_markup=main_menu())


# ==========================================
# BARCHA TESTLAR RO'YXATI VA BOSHLASH
# ==========================================

@dp.message(F.text == "📝 Testni boshlash")
async def select_quiz_set(message: Message):
    global_tests_data = load_tests()  # Yangi saqlangan testlarni yuklash

    if not global_tests_data:
        await message.answer("❌ Hali hech kim test qo'shmagan.", parse_mode="HTML")
        return

    buttons = []
    for code, data in global_tests_data.items():
        is_owner = " (Mening testim)" if data["author"] == message.from_user.id else ""
        btn_text = f"📁 {data['name']} ({len(data['questions'])} ta){is_owner}"
        buttons.append([InlineKeyboardButton(text=btn_text, callback_data=f"startset_{code}")])

    keyboard = InlineKeyboardMarkup(inline_keyboard=buttons)
    await message.answer("📝 <b>Barcha mavjud testlar ro'yxati:</b>", reply_markup=keyboard, parse_mode="HTML")


@dp.callback_query(F.data.startswith("startset_"))
async def start_quiz_from_set(callback: CallbackQuery, state: FSMContext):
    code = callback.data.split("startset_")[1]
    user_id = callback.from_user.id

    global_tests_data = load_tests()

    if code not in global_tests_data:
        await callback.answer("Test to'plami topilmadi.", show_alert=True)
        return

    test_data = global_tests_data[code]
    active_quizzes[user_id] = {
        "questions": test_data["questions"],
        "current": 0,
        "correct": 0,
        "wrong": 0,
        "timer_task": None
    }

    await state.set_state(TestState.answering)
    await callback.message.edit_text(f"🚀 <b>'{test_data['name']}' to'plami bo'yicha test boshlandi!</b>", parse_mode="HTML")
    await asyncio.sleep(1)
    await send_question_with_timer(user_id, state)


# ==========================================
# JAVOBLAR VA TESTNI YAKUNLASH
# ==========================================

@dp.callback_query(TestState.answering, F.data == "stop_quiz")
async def stop_quiz_handler(callback: CallbackQuery, state: FSMContext):
    await callback.message.edit_text("🛑 Test to'xtatildi...")
    await callback.answer()
    await show_final_results(callback.from_user.id, state, is_stopped=True)


@dp.callback_query(TestState.answering, F.data.startswith("ans_"))
async def process_answer(callback: CallbackQuery, state: FSMContext):
    user_id = callback.from_user.id
    selected_ans = callback.data.split("_")[1]

    quiz = active_quizzes.get(user_id)
    if not quiz:
        await callback.answer("Test topilmadi.", show_alert=True)
        return

    if quiz.get("timer_task"):
        quiz["timer_task"].cancel()

    current_idx = quiz["current"]
    q = quiz["questions"][current_idx]
    correct_ans = q["correct"]

    if selected_ans == correct_ans:
        quiz["correct"] += 1
        res_text = f"✅ <b>To'g'ri javob!</b>\n\nSiz tanladingiz: {selected_ans}"
    else:
        quiz["wrong"] += 1
        res_text = f"❌ <b>Noto'g'ri javob!</b>\n\nSiz tanladingiz: {selected_ans}\nTo'g'ri javob: <b>{correct_ans}</b>"

    quiz["current"] += 1

    await callback.message.edit_text(res_text, parse_mode="HTML")
    await callback.answer()

    await asyncio.sleep(1.5)
    await send_question_with_timer(user_id, state)


async def main():
    print("BOT ISHGA TUSHDI...")
    await dp.start_polling(bot)


if __name__ == "__main__":
    asyncio.run(main())

BOT ISHGA TUSHDI...
